In [1]:
# --- headless execution harness (added by run_experiments.py) ---
import os as _h_os
import matplotlib as _h_mpl
_h_mpl.use('Agg')
import matplotlib.pyplot as _h_plt
_OUT = 'C:\\Users\\ASUS\\group_sgu26_deep_learning\\BTL03\\output\\0. Getting Started\\Iris_Bayesian_decision_making_pytorch'
_h_os.makedirs(_OUT, exist_ok=True)
_h_n = [0]
_h_show = _h_plt.show
def _saving_show(*args, **kwargs):
    _figs = [_h_plt.figure(n) for n in _h_plt.get_fignums()]
    for _f in _figs:
        _h_n[0] += 1
        _f.savefig(_h_os.path.join(_OUT, 'fig_%02d.png' % _h_n[0]), dpi=110, bbox_inches='tight')
    _h_show(*args, **kwargs)
    for _f in _figs:
        _h_plt.close(_f)
_h_plt.show = _saving_show
print('[harness] figures -> ' + _OUT)


[harness] figures -> C:\Users\ASUS\group_sgu26_deep_learning\BTL03\output\0. Getting Started\Iris_Bayesian_decision_making_pytorch


# (1) Import libraries
- Notebook gốc sử dụng `scikit-learn` (`GaussianNB`) và `numpy` (không dùng Keras).
- Giữ nguyên toàn bộ phần gốc ở mục (1)–(4), đồng thời bổ sung mục (5) minh họa triển khai tương đương bằng **PyTorch Tensors**.

In [2]:
from sklearn import datasets
from sklearn.naive_bayes import GaussianNB

import numpy as np
import time
import torch

# (2) Preparing Iris dataset

In [3]:
iris = datasets.load_iris()

# get training and testing indexes randomly (60% train / 40% test)
train_indx = np.random.choice(range(0, 150), int(iris.target.shape[0] / 100 * 60), replace=False)
test_indx = np.array([indx for indx in np.arange(0, 150, 1) if indx not in train_indx])

x_train = iris.data[train_indx, :]
y_train = iris.target[train_indx]

x_test = iris.data[test_indx, :]
y_test = iris.target[test_indx]

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(90, 4)
(90,)
(60, 4)
(60,)


# (3) Parameters estimation

In [4]:
mu_0 = np.mean(x_train[y_train == 0], axis=0)
sigma_0 = np.cov(x_train[y_train == 0], rowvar=False)
print('Class 0 mean:', mu_0)
print('Class 0 cov:\n', sigma_0)

mu_1 = np.mean(x_train[y_train == 1], axis=0)
sigma_1 = np.cov(x_train[y_train == 1], rowvar=False)
print('Class 1 mean:', mu_1)
print('Class 1 cov:\n', sigma_1)

mu_2 = np.mean(x_train[y_train == 2], axis=0)
sigma_2 = np.cov(x_train[y_train == 2], rowvar=False)
print('Class 2 mean:', mu_2)
print('Class 2 cov:\n', sigma_2)

Class 0 mean: [5.03225806 3.44516129 1.42903226 0.24516129]
Class 0 cov:
 [[0.13892473 0.12416129 0.01503226 0.01082796]
 [0.12416129 0.18722581 0.00997849 0.00922581]
 [0.01503226 0.00997849 0.0347957  0.00664516]
 [0.01082796 0.00922581 0.00664516 0.01322581]]
Class 1 mean: [5.88064516 2.78387097 4.26451613 1.3516129 ]
Class 1 cov:
 [[0.26294624 0.06901075 0.19629032 0.05469892]
 [0.06901075 0.06473118 0.07307527 0.03186022]
 [0.19629032 0.07307527 0.25903226 0.08055914]
 [0.05469892 0.03186022 0.08055914 0.03658065]]
Class 2 mean: [6.58571429 3.00357143 5.59642857 2.075     ]
Class 2 cov:
 [[0.47089947 0.09227513 0.36624339 0.02333333]
 [0.09227513 0.11294974 0.07001323 0.03009259]
 [0.36624339 0.07001323 0.36998677 0.04027778]
 [0.02333333 0.03009259 0.04027778 0.05824074]]


# (4) Bayesian decision making (scikit-learn GaussianNB)

In [5]:
gnb = GaussianNB()

y_pred = gnb.fit(x_train, y_train).predict(x_test)
print('Testing classification accuracy (sklearn GaussianNB): %.4f' % ((y_test == y_pred).sum() / x_test.shape[0]))

Testing classification accuracy (sklearn GaussianNB): 0.9333


# (5) PyTorch Tensor Implementation (Gaussian Naive Bayes)

In [6]:
# Chuyển dữ liệu sang PyTorch Tensor
x_train_t = torch.tensor(x_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

classes = torch.unique(y_train_t)
log_priors = []
means = []
vars_ = []
eps = 1e-9

for c in classes:
    x_c = x_train_t[y_train_t == c]
    log_priors.append(torch.log(torch.tensor(len(x_c) / len(x_train_t), dtype=torch.float32)))
    means.append(x_c.mean(dim=0))
    vars_.append(x_c.var(dim=0, unbiased=False) + eps)

# Tính log-posterior cho từng mẫu test
log_posteriors = []
for i in range(len(classes)):
    dist = torch.distributions.Normal(means[i], torch.sqrt(vars_[i]))
    log_likelihood = dist.log_prob(x_test_t).sum(dim=1)
    log_posteriors.append(log_priors[i] + log_likelihood)

log_posteriors = torch.stack(log_posteriors, dim=1)
y_pred_torch = torch.argmax(log_posteriors, dim=1)

acc_torch = (y_pred_torch == y_test_t).float().mean().item()
print('Testing classification accuracy (PyTorch GaussianNB): %.4f' % acc_torch)

Testing classification accuracy (PyTorch GaussianNB): 0.9333
